# OSL-Words Model Evaluation
Loads the best checkpoint and evaluates it on the dev and test splits.
Reports `top1_acc_pi` (per-instance) and `top1_acc_pc` (per-class) accuracy, and shows sample predictions.

In [1]:
import os, sys
from pathlib import Path

# ── Paths ──────────────────────────────────────────────────────────────────────
PROJECT_ROOT = Path(r"C:\Users\MOBPC\Downloads\FYP\FYPproject")
UNISIGN_DIR  = PROJECT_ROOT / "Uni-Sign-main" / "Uni-Sign-main"
CHECKPOINT   = UNISIGN_DIR / "out" / "osl_words_finetuning" / "best_checkpoint.pth"

assert CHECKPOINT.exists(), f"Checkpoint not found: {CHECKPOINT}"
print(f"Checkpoint size: {CHECKPOINT.stat().st_size / 1e9:.2f} GB")

# Switch working directory so that relative paths in config.py resolve correctly
os.chdir(UNISIGN_DIR)
if str(UNISIGN_DIR) not in sys.path:
    sys.path.insert(0, str(UNISIGN_DIR))
print("Working directory:", os.getcwd())

Checkpoint size: 3.89 GB
Working directory: C:\Users\MOBPC\Downloads\FYP\FYPproject\Uni-Sign-main\Uni-Sign-main


In [1]:
import torch
from torch.nn.utils.rnn import pad_sequence
from torch.utils.data import DataLoader

from models import Uni_Sign
from datasets import S2T_Dataset
from SLRT_metrics import islr_performance
from utils import get_args_parser
from config import test_label_paths, dev_label_paths

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)
if DEVICE.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))

ModuleNotFoundError: No module named 'models'

In [3]:
# ── Build args (same settings used during training) ────────────────────────────
parser = get_args_parser()
args = parser.parse_args([
    "--dataset",    "OSL-Words",
    "--task",       "ISLR",
    "--batch-size", "4",
    "--max_length", "256",
    "--num_workers","0",         # must be 0 on Windows inside Jupyter
    "--no-pin-mem",              # safer without pinned memory on Windows
    "--eval",                    # flag used to signal evaluation mode
])
args.distributed = False
args.finetune = str(CHECKPOINT)
print(args)

Namespace(batch_size=4, gradient_accumulation_steps=8, gradient_clipping=1.0, epochs=20, start_epoch=0, world_size=1, dist_url='env://', local_rank=0, hidden_dim=256, finetune='C:\\Users\\MOBPC\\Downloads\\FYP\\FYPproject\\Uni-Sign-main\\Uni-Sign-main\\out\\osl_words_finetuning\\best_checkpoint.pth', opt='adamw', opt_eps=1e-09, opt_betas=None, clip_grad=None, momentum=0.9, weight_decay=0.0001, sched='cosine', lr=0.001, min_lr=1e-08, warmup_epochs=0, output_dir='', seed=42, eval=True, num_workers=0, pin_mem=False, offload=False, dtype='bf16', zero_stage=2, compute_fp32_loss=False, quick_break=0, rgb_support=False, max_length=256, dataset='OSL-Words', task='ISLR', label_smoothing=0.2, online_video='', distributed=False)


In [4]:
# ── DataLoaders ────────────────────────────────────────────────────────────────
print("Loading dev split ...")
dev_data = S2T_Dataset(path=dev_label_paths["OSL-Words"], args=args, phase="dev")
dev_loader = DataLoader(
    dev_data,
    batch_size=args.batch_size,
    num_workers=args.num_workers,
    collate_fn=dev_data.collate_fn,
    sampler=torch.utils.data.SequentialSampler(dev_data),
    pin_memory=args.pin_mem,
)
print(f"Dev samples : {len(dev_data)}")

print("Loading test split ...")
test_data = S2T_Dataset(path=test_label_paths["OSL-Words"], args=args, phase="test")
test_loader = DataLoader(
    test_data,
    batch_size=args.batch_size,
    num_workers=args.num_workers,
    collate_fn=test_data.collate_fn,
    sampler=torch.utils.data.SequentialSampler(test_data),
    pin_memory=args.pin_mem,
)
print(f"Test samples: {len(test_data)}")

Loading dev split ...
Dev samples : 185
Loading test split ...
Test samples: 34


In [5]:
# ── Build & load model ─────────────────────────────────────────────────────────
print("Building model ...")
model = Uni_Sign(args=args).to(DEVICE)
model.eval()

print(f"Loading checkpoint: {CHECKPOINT}")
ckpt = torch.load(str(CHECKPOINT), map_location="cpu")
state_dict = ckpt["model"]
ret = model.load_state_dict(state_dict, strict=False)
print("Missing keys   :", ret.missing_keys)
print("Unexpected keys:", ret.unexpected_keys)
del ckpt, state_dict
if DEVICE.type == "cuda":
    torch.cuda.empty_cache()
print("Model ready.")

Building model ...


Exception in thread Thread-auto_conversion:
Traceback (most recent call last):
  File "c:\Users\MOBPC\anaconda3\envs\torchgpu\lib\threading.py", line 1016, in _bootstrap_inner
    self.run()
  File "c:\Users\MOBPC\anaconda3\envs\torchgpu\lib\threading.py", line 953, in run
    self._target(*self._args, **self._kwargs)
  File "c:\Users\MOBPC\anaconda3\envs\torchgpu\lib\site-packages\transformers\safetensors_conversion.py", line 116, in auto_conversion
    raise e
  File "c:\Users\MOBPC\anaconda3\envs\torchgpu\lib\site-packages\transformers\safetensors_conversion.py", line 95, in auto_conversion
    sha = get_conversion_pr_reference(api, pretrained_model_name_or_path, **cached_file_kwargs)
  File "c:\Users\MOBPC\anaconda3\envs\torchgpu\lib\site-packages\transformers\safetensors_conversion.py", line 71, in get_conversion_pr_reference
    spawn_conversion(token, private, model_id)
  File "c:\Users\MOBPC\anaconda3\envs\torchgpu\lib\site-packages\transformers\safetensors_conversion.py", line

Loading weights:   0%|          | 0/284 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to encoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie shared.weight to decoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


Loading checkpoint: C:\Users\MOBPC\Downloads\FYP\FYPproject\Uni-Sign-main\Uni-Sign-main\out\osl_words_finetuning\best_checkpoint.pth
Missing keys   : []
Unexpected keys: []
Model ready.


In [6]:
# ── Inference helper ───────────────────────────────────────────────────────────
def run_inference(loader, split_name):
    """Run beam-search inference and return (refs, preds, names)."""
    tgt_refs, tgt_pres, tgt_names = [], [], []
    tokenizer = model.mt5_tokenizer
    padding_value = tokenizer.eos_token_id

    model.eval()
    with torch.no_grad():
        for step, (src_input, tgt_input) in enumerate(loader):
            # Move pose tensors to device
            for key in src_input:
                if isinstance(src_input[key], torch.Tensor):
                    src_input[key] = src_input[key].float().to(DEVICE)

            # Forward pass (also computes loss internally)
            stack_out = model(src_input, tgt_input)

            # Beam-search decoding
            output = model.generate(stack_out, max_new_tokens=100, num_beams=4)

            for i in range(len(output)):
                tgt_pres.append(output[i])
                tgt_refs.append(tgt_input["gt_sentence"][i])
                tgt_names.append(src_input["name_batch"][i])

            if (step + 1) % 10 == 0:
                print(f"  [{split_name}] {step+1}/{len(loader)} batches done")

    # Decode token IDs → strings
    pad_tensor = torch.ones(150 - len(tgt_pres[0])).to(DEVICE) * padding_value
    tgt_pres[0] = torch.cat((tgt_pres[0], pad_tensor.long()), dim=0)
    tgt_pres = pad_sequence(tgt_pres, batch_first=True, padding_value=padding_value)
    tgt_pres = tokenizer.batch_decode(tgt_pres, skip_special_tokens=True)

    return tgt_refs, tgt_pres, tgt_names

In [7]:
# ── Evaluate on DEV set ────────────────────────────────────────────────────────
print("=" * 50)
print("Running inference on DEV set ...")
print("=" * 50)
dev_refs, dev_preds, dev_names = run_inference(dev_loader, "dev")

dev_pi, dev_pc = islr_performance(dev_refs, dev_preds)
print(f"\n📊 DEV Results:")
print(f"   top1_acc_pi (per-instance) : {dev_pi:.2f}%")
print(f"   top1_acc_pc (per-class)    : {dev_pc:.2f}%")

Running inference on DEV set ...
  [dev] 10/47 batches done
  [dev] 20/47 batches done
  [dev] 30/47 batches done
  [dev] 40/47 batches done
top1_acc_pi: 90.81
top1_acc_pc: 90.76

📊 DEV Results:
   top1_acc_pi (per-instance) : 90.81%
   top1_acc_pc (per-class)    : 90.76%


In [8]:
# ── Evaluate on TEST set ───────────────────────────────────────────────────────
print("=" * 50)
print("Running inference on TEST set ...")
print("=" * 50)
test_refs, test_preds, test_names = run_inference(test_loader, "test")

test_pi, test_pc = islr_performance(test_refs, test_preds)
print(f"\n📊 TEST Results:")
print(f"   top1_acc_pi (per-instance) : {test_pi:.2f}%")
print(f"   top1_acc_pc (per-class)    : {test_pc:.2f}%")

Running inference on TEST set ...
top1_acc_pi: 88.24
top1_acc_pc: 87.88

📊 TEST Results:
   top1_acc_pi (per-instance) : 88.24%
   top1_acc_pc (per-class)    : 87.88%


In [9]:
# ── Summary table ──────────────────────────────────────────────────────────────
print("\n" + "=" * 50)
print(f"{'Split':<8} {'top1_acc_pi':>14} {'top1_acc_pc':>14}")
print("-" * 38)
print(f"{'Dev':<8} {dev_pi:>13.2f}% {dev_pc:>13.2f}%")
print(f"{'Test':<8} {test_pi:>13.2f}% {test_pc:>13.2f}%")
print("=" * 50)


Split       top1_acc_pi    top1_acc_pc
--------------------------------------
Dev              90.81%         90.76%
Test             88.24%         87.88%


In [10]:
# ── Sample predictions (first 20 from test set) ────────────────────────────────
N = min(20, len(test_refs))
print(f"{'#':<4} {'Name':<30} {'Ground Truth':<25} {'Prediction':<25} {'✓':<3}")
print("-" * 90)
for i in range(N):
    correct = "✓" if test_refs[i].strip() == test_preds[i].strip() else "✗"
    name = test_names[i][-28:] if len(test_names[i]) > 28 else test_names[i]
    print(f"{i+1:<4} {name:<30} {test_refs[i]:<25} {test_preds[i]:<25} {correct:<3}")

#    Name                           Ground Truth              Prediction                ✓  
------------------------------------------------------------------------------------------
1    0005_S02_T01                   ممرضة                     ممرضة                     ✓  
2    0029_S02_T02                   خذ دواء                   خذ دواء                   ✓  
3    0052_S07_T01                   ينجح                      لأول مرة                  ✗  
4    0045_S01_T02                   يقرأ                      يقرأ                      ✓  
5    0344_S05_T04                   سيئ                       سيئ                       ✓  
6    0008_S02_T02                   ألم                       ألم                       ✓  
7    0031_S02_T02                   مدرسة                     مدرسة                     ✓  
8    0345_S07_T01                   مركز                      يمشي                      ✗  
9    0038_S01_T02                   دفتر                      دفتر               